# MINLP Algorithms

Integer programs are easy to write and hard to solve. The handout on *Integer Programming Algorithms*
makes the point with one table: complete enumeration of 35 binary variables takes about 110 years at a
tenth of a second per LP, and the Sudoku model from two lectures ago has 513 of them. Every algorithm in
this notebook exists to skip almost all of that search space and **prove** that nothing was lost.

This notebook is the computational companion to that handout. It does not re-derive the theory — the
handout does that — it runs the two algorithms and shows you what they look like from the inside:

1. **Branch and bound**, as a loop rather than as nine cells of hand-fixed variables, on the handout's
   worked example. The output is the search tree.
2. **Outer approximation**, implemented from the handout's flowchart — an NLP subproblem for the upper
   bound, an MILP master problem for the lower bound, integer cuts, and a gap that has to close — on a
   convex MINLP for the design of multilayer building insulation.

:::{note}
The companion notebook [](../8/MILP.ipynb) walks the same branch and bound example node by node, one cell
per node. Read that first if you want to see each LP relaxation on its own. Here the point is the
algorithm, not the nodes.
:::

In [ ]:
# This code cell installs packages on Colab

import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper

helper.set_plotting_style()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

# House figure style. On Colab only the notebook is present, so fall back to the raw URL.
STYLE = (
    "https://raw.githubusercontent.com/ndcbe/optimization/main/figures/dowling.mplstyle"
    if "google.colab" in sys.modules
    else "../../figures/dowling.mplstyle"
)
plt.style.use(STYLE)

solver_milp = pyo.SolverFactory("appsi_highs")
solver_nlp = pyo.SolverFactory("ipopt")

## 1. Branch and bound

$$
\begin{aligned}
\min_{x,\,y} \quad & z = x + y_1 + 3y_2 + 2y_3 \\
\text{s.t.} \quad & -x + 3y_1 + 2y_2 + y_3 \le 0 \\
                  & -5y_1 - 8y_2 - 3y_3 \le -9 \\
                  & x \ge 0, \qquad y \in \{0,1\}^{3}
\end{aligned}
$$

This is problem (MIPEX) from Biegler, Grossmann & Westerberg (1997), §A.3.2, p. 760, and the tree we
build below is their Figure A.10.

The handout's algorithm, in three cases. At each node solve the LP relaxation, then:

* **case 1** — the LP is infeasible, or its value already exceeds the incumbent: *fathom* the node and
  everything below it;
* **case 2** — the LP solution is integral: it is optimal for that subtree, so record it as the incumbent
  and stop descending;
* **case 3** — otherwise, branch on a fractional variable.

Only case 3 grows the tree. The efficiency of the whole method is the proportion of nodes settled by
cases 1 and 2.

In [ ]:
def build_mipex_relaxation(fixed=None):
    """LP relaxation of (MIPEX) with some binaries fixed.

    Arguments:
        fixed: dict mapping binary index (1, 2, 3) to the value it is fixed at

    Returns:
        a Pyomo ConcreteModel
    """
    m = pyo.ConcreteModel("MIPEX LP relaxation")
    m.J = pyo.RangeSet(1, 3)

    m.x = pyo.Var(domain=pyo.NonNegativeReals)
    m.y = pyo.Var(m.J, bounds=(0, 1))  # integrality relaxed to the unit box

    @m.Constraint()
    def coupling(m):
        return -m.x + 3 * m.y[1] + 2 * m.y[2] + m.y[3] <= 0

    @m.Constraint()
    def covering(m):
        return -5 * m.y[1] - 8 * m.y[2] - 3 * m.y[3] <= -9

    @m.Objective(sense=pyo.minimize)
    def z(m):
        return m.x + m.y[1] + 3 * m.y[2] + 2 * m.y[3]

    for j, value in (fixed or {}).items():
        m.y[j].fix(value)

    return m


def solve_node(fixed):
    """Solve one node's LP relaxation and report what happened.

    An infeasible relaxation is an ANSWER here, not a failure -- it fathoms a whole
    subtree. So the termination condition is checked explicitly and `load_solutions`
    is deferred: HiGHS raises if asked to load a solution that does not exist.
    """
    m = build_mipex_relaxation(fixed)
    results = solver_milp.solve(m, load_solutions=False)
    tc = results.solver.termination_condition

    if pyo.check_optimal_termination(results):
        m.solutions.load_from(results)
        return {
            "status": "optimal",
            "z": pyo.value(m.z),
            "y": np.array([pyo.value(m.y[j]) for j in m.J]),
            "x": pyo.value(m.x),
        }
    if tc in (
        pyo.TerminationCondition.infeasible,
        pyo.TerminationCondition.infeasibleOrUnbounded,
    ):
        return {"status": "infeasible", "z": None, "y": None, "x": None}

    raise RuntimeError(f"unexpected termination condition at node {fixed}: {tc}")

### The search

The handout works **breadth first** — every node of one level before descending — matching BGW's Figure
A.10. A breadth-first traversal is a **FIFO queue** of active nodes: `pop(0)` takes the oldest, and
children are appended to the back. Change that one line to `pop()` and you have depth first.

In [ ]:
INT_TOL = 1e-6


def branch_and_bound(verbose=True):
    """Breadth-first branch and bound on (MIPEX).

    Returns:
        nodes: list of dicts, one per node, in the order the LPs were solved
        edges: list of (parent, child, label) tuples describing the tree
        incumbent: dict with the optimal solution
    """
    incumbent_z, incumbent = np.inf, None
    queue = [{"fixed": {}, "parent": None, "label": ""}]  # FIFO -> breadth first
    nodes, edges = [], []
    node_id = 0

    while queue:
        node = queue.pop(0)
        node_id += 1
        result = solve_node(node["fixed"])
        if node["parent"] is not None:
            edges.append((node["parent"], node_id, node["label"]))

        record = {
            "node": node_id,
            "fixed": ", ".join(f"y{j}={v}" for j, v in node["fixed"].items()) or "--",
            "z_LP": result["z"],
            "y_LP": None if result["y"] is None else np.round(result["y"], 4),
        }

        # case 1a: the relaxation has no feasible point, so neither does the subtree
        if result["status"] == "infeasible":
            record.update(case=1, outcome="infeasible -> fathom")

        # case 1b: the bound is already worse than a solution we hold in hand
        elif result["z"] > incumbent_z - INT_TOL:
            record.update(
                case=1, outcome=f"bound {result['z']:.2f} >= incumbent -> fathom"
            )

        else:
            fractional = [
                j
                for j in (1, 2, 3)
                if abs(result["y"][j - 1] - round(result["y"][j - 1])) > INT_TOL
            ]
            # case 2: the relaxation happened to land on an integer point
            if not fractional:
                incumbent_z, incumbent = result["z"], {**result, "node": node_id}
                record.update(
                    case=2, outcome=f"integral -> incumbent z = {result['z']:.0f}"
                )
            # case 3: branch
            else:
                j = fractional[0]
                record.update(case=3, outcome=f"branch on y{j}")
                for value in (0, 1):
                    queue.append(
                        {
                            "fixed": {**node["fixed"], j: value},
                            "parent": node_id,
                            "label": f"$y_{j}$={value}",
                        }
                    )

        nodes.append(record)
        if verbose:
            z_text = (
                "infeasible" if record["z_LP"] is None else f"{record['z_LP']:8.4f}"
            )
            print(
                f"node {node_id}: fixed {record['fixed']:<20s} z_LP = {z_text}   {record['outcome']}"
            )

    return nodes, edges, incumbent


nodes, edges, incumbent = branch_and_bound()

In [ ]:
table = pd.DataFrame(nodes).set_index("node")
display(table)

print(f"\nz*  = {incumbent['z']:.0f}")
print(f"y*  = {np.round(incumbent['y']).astype(int)}")
print(f"x*  = {incumbent['x']:.0f}   (found at node {incumbent['node']})")
print(f"\n{len(nodes)} nodes examined out of the 15 in the full tree")

Nine nodes, and the numbers match the handout line for line: root value $5.8$ at $(0.2, 1, 0)$, nodes 4,
6 and 8 infeasible, an incumbent of $9$ at node 7, and the optimum $z^* = 8$ at $y^* = (0,1,1)$,
$x^* = 3$ at node 9. Only 2 of the 8 leaves were ever reached.

Three things to notice, all of them the handout's:

* **Node 6 is infeasible because the *relaxation* is.** With $y_1 = 1$, $y_2 = 0$ the covering constraint
  needs $3y_3 \ge 4$, i.e. $y_3 \ge 4/3 > 1$. No integer point below it can be feasible either.
* **Node 7's incumbent of 9 is what makes case 1 available at all** for the rest of the search. Before
  node 7 there is nothing to compare a bound against.
* **Node 5 survives**, with $z_{LP} = 6.75 < 9$ — and it had to, because the optimum lies below it. A
  valid bound can never fathom the optimum.

What the traversal order changes is the **work**. Descending the $y_1 = 0$ branch depth first would have
found $z_\ell = 8$ before ever solving node 7, whose $z_{LP} = 9 > 8$ would then have been fathomed by
case 1 instead of becoming an incumbent. Finding a good incumbent early is worth as much as a good bound.

### The search tree

The tree the loop above just built, drawn from the `edges` it returned rather than typed in by hand.

In [ ]:
def plot_tree(nodes, edges, ax=None):
    """Draw the branch and bound tree from the node records and edge list."""
    table = {n["node"]: n for n in nodes}

    depth = {1: 0}
    children = {}
    for parent, child, _ in edges:
        depth[child] = depth[parent] + 1
        children.setdefault(parent, []).append(child)

    # tidy layout: leaves get consecutive slots left to right, a parent sits
    # at the midpoint of its children
    x_of = {}
    next_slot = [0.0]

    def place(node_id):
        kids = children.get(node_id, [])
        if not kids:
            x_of[node_id] = next_slot[0]
            next_slot[0] += 1.0
        else:
            for kid in kids:
                place(kid)
            x_of[node_id] = 0.5 * (x_of[kids[0]] + x_of[kids[-1]])

    place(1)
    pos = {node_id: (x_of[node_id], -depth[node_id]) for node_id in depth}

    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.5))

    for parent, child, label in edges:
        (x0, y0), (x1, y1) = pos[parent], pos[child]
        ax.plot([x0, x1], [y0, y1], color="0.45", lw=1.5, ls="-", zorder=1)
        ax.annotate(
            label,
            xy=(x0 + 0.62 * (x1 - x0), y0 + 0.62 * (y1 - y0)),
            ha="center",
            va="center",
            fontsize=11,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none"),
            zorder=2,
        )

    for node_id, (x, y) in pos.items():
        record = table[node_id]
        # case 2 nodes (integer solutions) are filled; everything else is open
        filled = record["case"] == 2
        ax.plot(
            x,
            y,
            marker="o",
            ms=26,
            mfc="black" if filled else "white",
            mec="black",
            mew=1.8,
            zorder=3,
        )
        ax.annotate(
            str(node_id),
            xy=(x, y),
            ha="center",
            va="center",
            fontsize=13,
            color="white" if filled else "black",
            zorder=4,
        )
        text = "infeas." if record["z_LP"] is None else f"$z={record['z_LP']:g}$"
        ax.annotate(
            text, xy=(x + 0.20, y), ha="left", va="center", fontsize=12, zorder=4
        )

    ax.set_xlim(min(x_of.values()) - 0.6, max(x_of.values()) + 0.9)
    ax.set_ylim(-max(depth.values()) - 0.5, 0.5)
    ax.axis("off")
    return ax


fig, ax = plt.subplots(figsize=(8.0, 4.8))
plot_tree(nodes, edges, ax=ax)
ax.set_title(
    "Branch and bound tree, breadth first\n(filled = integer solution)", fontsize=14
)
plt.show()

Node 9 is the optimum. Nodes 4, 6 and 8 are infeasible; nodes 7 and 9 are the two integer solutions found,
in that order. The three infeasible leaves and the fathomed subtrees are the $99.99\ldots\%$ the algorithm
skipped — six of the fifteen nodes here, and the difference between minutes and centuries at thirty-five
binaries.

Finally, the sanity check every implementation needs: hand the whole problem to a real MILP solver and
confirm it agrees.

In [ ]:
def build_mipex_milp():
    """(MIPEX) with the integrality restored."""
    m = build_mipex_relaxation()
    for j in m.J:
        m.y[j].domain = pyo.Binary
    return m


m = build_mipex_milp()
results = solver_milp.solve(m, load_solutions=False)
assert pyo.check_optimal_termination(
    results
), f"MILP solve failed: {results.solver.termination_condition}"
m.solutions.load_from(results)

print(
    f"HiGHS: z* = {pyo.value(m.z):.0f}, "
    f"y* = {[round(pyo.value(m.y[j])) for j in m.J]}, x* = {pyo.value(m.x):.0f}"
)
assert (
    abs(pyo.value(m.z) - incumbent["z"]) < 1e-6
), "our branch and bound disagrees with HiGHS"

## 2. Outer approximation

Branch and bound needs a relaxation that is cheap to solve. When the continuous part of the problem is
nonlinear, the LP relaxation is gone and something else has to supply the bound. **Outer approximation**
supplies it by linearizing.

The handout states the MINLP as

$$
\begin{aligned}
\min_{x,\,y} \quad & z = c^\top y + f(x) \\
\text{s.t.} \quad & g(x) + By \le 0, \qquad Ay \le a, \qquad x \in X, \qquad y \in \{0,1\}^{q},
\end{aligned}
$$

and alternates two problems:

* the **NLP subproblem**, with $y$ fixed at a guess $y^{k}$. It is a *restriction*, so every solution is
  an implementable design and its value $z^{u}$ is an **upper** bound.
* the **MILP master problem**, in which every nonlinear function is replaced by its first-order Taylor
  expansion at the points $x^{1}, \dots, x^{k}$ visited so far. If $f$ and $g$ are **convex**, each
  tangent plane underestimates the function it replaces, the master's feasible set contains the MINLP's,
  and its value $z^{L}$ is a **lower** bound.

Between iterations an **integer cut** excludes the $y^{k}$ just tried,

$$
\sum_{i \in B^{k}} y_i - \sum_{i \in N^{k}} y_i \le |B^{k}| - 1,
\qquad B^{k} = \{i : y^{k}_i = 1\}, \quad N^{k} = \{i : y^{k}_i = 0\},
$$

and the loop stops when $z^{u} - z^{L} \le \varepsilon$.

**Convexity is not optional.** If $f$ or $g$ is nonconvex a tangent plane can cut into the feasible
region, $z^{L}$ is not a bound, and the method can converge to the wrong answer. The example below is
chosen so that the requirement is satisfied, and we check it.

### The example: multilayer building insulation

A wall separates conditioned interior space from the environment. Heat flows through it at a rate
proportional to the overall heat transfer coefficient $U = 1/R$, where the thermal resistance of $N$
insulating layers adds in series:

$$
R = R_0 + \sum_{n=1}^{N} \frac{x_n}{k_n},
$$

with $x_n$ the thickness of layer $n$ [m], $k_n$ its thermal conductivity [W/m/K], and $R_0$ the
resistance of the structural elements. Annual energy cost is proportional to $U$; installing layer $n$
costs a fixed $a_n$ plus $b_n$ per unit thickness. So with $y_n \in \{0,1\}$ indicating whether layer $n$
is installed at all,

$$
\begin{aligned}
\min_{x,\,y} \quad & \frac{\alpha}{R_0 + \sum_n x_n / k_n}
                     \;+\; \beta \sum_n \left( a_n y_n + b_n x_n \right) \\
\text{s.t.} \quad & x_n \le t_n^{\max} y_n && n = 1, \dots, N \\
                  & \sum_n x_n \le T \\
                  & x_n \ge 0, \qquad y_n \in \{0,1\}.
\end{aligned}
$$

Read it against the handout's form. The fixed installation charges $\beta a_n y_n$ are $c^\top y$; the
energy and material costs are $f(x)$; the rows $x_n \le t_n^{\max} y_n$ are $g(x) + By \le 0$, the
constraints that couple the two kinds of variable; the thickness budget involves $x$ only. There are no
pure-binary rows $Ay \le a$ in this instance.

**$f$ is convex.** $\alpha/R$ is a convex decreasing function of $R > 0$ composed with an affine,
increasing function of $x$, and the rest is linear. Every coupled constraint is already linear. So outer
approximation is on firm ground here — and Ipopt will find the *global* optimum of each NLP subproblem,
not merely a local one.

:::{note}
**Source and licence.** The application, the physical model and the material data are adapted from
*Hands-On Mathematical Optimization with Python* by Postek, Zocca, Gromicho and Kantor (Cambridge
University Press, 2025), notebook 6.4, "Optimal Design of Multilayered Building Insulation". The MO-book
code is MIT-licensed, Copyright (c) 2022 Jeffrey Kantor, and that notice travels with substantial portions
of it. Their notebook solves the problem as a mixed-integer *conic* program with Mosek; here it is written
in the handout's MINLP form and solved by outer approximation, which is a different algorithm reaching the
same answer. The per-layer thickness limits $t_n^{\max}$ and the fifth material are additions of this
course, so that more than one layer is worth installing.
:::

In [ ]:
# k: thermal conductivity [W/m/K]; a: fixed installation cost [$/m^2];
# b: installed material cost [$/m^3]; tmax: largest available thickness [m]
materials = pd.DataFrame(
    {
        "Fiberglass batt": {"k": 0.040, "a": 4.0, "b": 60.0, "tmax": 0.06},
        "Mineral wool": {"k": 0.030, "a": 5.0, "b": 150.0, "tmax": 0.06},
        "Rigid foam (low R)": {"k": 0.030, "a": 8.0, "b": 120.0, "tmax": 0.05},
        "Rigid foam (high R)": {"k": 0.015, "a": 8.0, "b": 180.0, "tmax": 0.05},
        "Aerogel blanket": {"k": 0.013, "a": 12.0, "b": 900.0, "tmax": 0.02},
    }
).T

ALPHA = 60.0  # annualized energy cost per unit U   [$ K / W / m^2]
BETA = 0.05  # equivalent annual cost factor on capital
R0 = 2.0  # resistance of the structural elements  [m^2 K / W]
T_TOTAL = 0.15  # total thickness the wall cavity allows [m]

display(materials)

Two derived quantities explain most of the answer in advance. A material's **cost per unit of thermal
resistance** is $b_n k_n$, and its **resistance per unit thickness** is $1/k_n$. Cheapest is not thinnest,
which is exactly why the problem is interesting: the thickness budget $T$ forces a trade.

In [ ]:
economics = pd.DataFrame(
    {
        "cost per unit R  [$ W / K]": materials["b"] * materials["k"],
        "R per unit thickness  [K/W/m]": 1.0 / materials["k"],
        "fixed charge  [$/m^2/yr]": BETA * materials["a"],
    }
)
display(economics.round(2))

In [ ]:
def thermal_resistance(x):
    """R = R0 + sum_n x_n / k_n for a dict of thicknesses."""
    return R0 + sum(x[n] / materials.loc[n, "k"] for n in materials.index)


def f_continuous(x):
    """The nonlinear part of the objective, f(x): energy cost plus material cost."""
    return ALPHA / thermal_resistance(x) + BETA * sum(
        materials.loc[n, "b"] * x[n] for n in materials.index
    )


def grad_f_continuous(x):
    """Gradient of f at x. d/dx_n [alpha / R] = -alpha / R^2 / k_n."""
    R = thermal_resistance(x)
    return {
        n: -ALPHA / R**2 / materials.loc[n, "k"] + BETA * materials.loc[n, "b"]
        for n in materials.index
    }

### The NLP subproblem: fix $y$, get an upper bound

In [ ]:
def build_nlp_subproblem(y):
    """NLP subproblem of outer approximation: the design cost with the layers y fixed.

    Arguments:
        y: dict mapping material name to 0 or 1

    Returns:
        a Pyomo ConcreteModel
    """
    m = pyo.ConcreteModel("insulation NLP subproblem")
    m.N = pyo.Set(initialize=list(materials.index))
    m.x = pyo.Var(m.N, domain=pyo.NonNegativeReals, bounds=(0, T_TOTAL))

    @m.Constraint(m.N)
    def layer_available(m, n):
        return m.x[n] <= materials.loc[n, "tmax"] * y[n]

    @m.Constraint()
    def thickness_budget(m):
        return sum(m.x[n] for n in m.N) <= T_TOTAL

    @m.Objective(sense=pyo.minimize)
    def cost(m):
        R = R0 + sum(m.x[n] / materials.loc[n, "k"] for n in m.N)
        return (
            ALPHA / R
            + BETA * sum(materials.loc[n, "b"] * m.x[n] for n in m.N)
            + BETA
            * sum(materials.loc[n, "a"] * y[n] for n in m.N)  # c^T y, a constant here
        )

    return m


def solve_nlp_subproblem(y):
    """Solve the NLP subproblem and return (x^k, z^u)."""
    m = build_nlp_subproblem(y)
    results = solver_nlp.solve(m)
    assert pyo.check_optimal_termination(
        results
    ), f"NLP subproblem failed for y = {y}: {results.solver.termination_condition}"
    return {n: pyo.value(m.x[n]) for n in materials.index}, pyo.value(m.cost)

### The MILP master problem: linearize, get a lower bound

The master problem carries **one copy of the linearized objective row for every point visited so far**,
not just the latest. That accumulation is what makes $z^{L}$ increase monotonically and the method
terminate: each new NLP solution adds a tangent plane that cuts off the design just tried.

In [ ]:
def build_master_problem(points, tried):
    """MILP master problem of outer approximation.

    Arguments:
        points: list of x^k dicts, one per iteration so far -- the linearization points
        tried:  list of y^k dicts already evaluated, each excluded by an integer cut

    Returns:
        a Pyomo ConcreteModel
    """
    m = pyo.ConcreteModel("outer approximation master MILP")
    m.N = pyo.Set(initialize=list(materials.index))
    m.x = pyo.Var(m.N, domain=pyo.NonNegativeReals, bounds=(0, T_TOTAL))
    m.y = pyo.Var(m.N, domain=pyo.Binary)
    m.epi = pyo.Var(domain=pyo.Reals)  # the epigraph variable, alpha in the handout

    @m.Constraint(m.N)
    def layer_available(m, n):
        return m.x[n] <= materials.loc[n, "tmax"] * m.y[n]

    @m.Constraint()
    def thickness_budget(m):
        return sum(m.x[n] for n in m.N) <= T_TOTAL

    # one outer approximation of the objective per point visited: since f is convex,
    # every tangent plane lies below f, so each row is a valid relaxation
    m.outer_approximation = pyo.ConstraintList()
    for xk in points:
        fk = f_continuous(xk)
        gk = grad_f_continuous(xk)
        m.outer_approximation.add(
            m.epi
            >= BETA * sum(materials.loc[n, "a"] * m.y[n] for n in m.N)
            + fk
            + sum(gk[n] * (m.x[n] - xk[n]) for n in m.N)
        )

    # integer cuts: at least one component of y must differ from each y^k already tried
    m.integer_cuts = pyo.ConstraintList()
    for yk in tried:
        ones = [n for n in m.N if yk[n] > 0.5]
        zeros = [n for n in m.N if yk[n] <= 0.5]
        m.integer_cuts.add(
            sum(m.y[n] for n in ones) - sum(m.y[n] for n in zeros) <= len(ones) - 1
        )

    @m.Objective(sense=pyo.minimize)
    def lower_bound(m):
        return m.epi

    return m

### The loop

Start from a deliberately poor guess: aerogel alone. It has the highest resistance per unit thickness of
anything in the table and by far the highest cost per unit of resistance — the kind of specification a
catalogue leads you to and an optimizer talks you out of.

In [ ]:
def outer_approximation(y_start, eps=1e-4, max_iter=20, verbose=True):
    """Outer approximation for the insulation MINLP, following the handout's flowchart.

    Arguments:
        y_start: dict of initial binary values
        eps: gap tolerance on z^u - z^L
        max_iter: iteration cap, purely a safety net

    Returns:
        history DataFrame, and the best (y, x, cost) found
    """
    y = dict(y_start)
    points, tried, history = [], [], []
    z_upper, best = np.inf, None
    z_lower = -np.inf

    for k in range(1, max_iter + 1):
        # --- NLP subproblem: a feasible design, hence an upper bound
        xk, z_nlp = solve_nlp_subproblem(y)
        if z_nlp < z_upper:
            z_upper, best = z_nlp, (dict(y), dict(xk))
        points.append(xk)
        tried.append(dict(y))

        # --- MILP master problem: a relaxation, hence a lower bound
        master = build_master_problem(points, tried)
        results = solver_milp.solve(master, load_solutions=False)
        if not pyo.check_optimal_termination(results):
            # every binary point has been cut off: nothing left to try
            assert (
                results.solver.termination_condition
                == pyo.TerminationCondition.infeasible
            ), f"master problem failed: {results.solver.termination_condition}"
            history.append(
                {
                    "iteration": k,
                    "z_nlp": z_nlp,
                    "z_upper": z_upper,
                    "z_lower": z_lower,
                    "layers": sum(y.values()),
                }
            )
            if verbose:
                print(f"iteration {k}: master infeasible -- search exhausted")
            break
        master.solutions.load_from(results)
        z_lower = pyo.value(master.lower_bound)

        history.append(
            {
                "iteration": k,
                "z_nlp": z_nlp,
                "z_upper": z_upper,
                "z_lower": z_lower,
                "layers": sum(y.values()),
            }
        )
        if verbose:
            print(
                f"iteration {k}: z^u = {z_upper:8.4f}   z^L = {z_lower:8.4f}   "
                f"gap = {z_upper - z_lower:8.4f}   "
                f"y = {[int(round(y[n])) for n in materials.index]}"
            )

        if z_upper - z_lower <= eps:
            break

        y = {n: round(pyo.value(master.y[n])) for n in materials.index}

    return pd.DataFrame(history).set_index("iteration"), best, z_upper


y_start = {n: (1 if n == "Aerogel blanket" else 0) for n in materials.index}
history, best, oa_cost = outer_approximation(y_start)

In [ ]:
display(history.round(4))

y_best, x_best = best
solution = materials.copy()
solution["installed"] = [int(y_best[n]) for n in materials.index]
solution["x opt [m]"] = [max(x_best[n], 0.0) for n in materials.index]
display(solution[["k", "a", "b", "tmax", "installed", "x opt [m]"]].round(5))

print(f"optimal annualized cost = {oa_cost:0.4f} $/m^2")
print(
    f"total thickness         = {sum(max(v, 0.0) for v in x_best.values()):0.4f} m "
    f"(budget {T_TOTAL} m)"
)
print(f"overall resistance R    = {thermal_resistance(x_best):0.3f} m^2 K / W")

Three iterations. Read the bounds column by column:

* **Iteration 1** evaluates the aerogel-only design. It is feasible, so $z^{u}$ is an honest upper bound —
  and an expensive one. The master problem, holding a single tangent plane, returns a lower bound far
  below anything achievable: one linearization is a very weak relaxation.
* **Iteration 2** jumps to a much better design. The lower bound leaps, because the master now has two
  tangent planes and an integer cut.
* **Iteration 3** finds the optimum, and the master's value rises above the incumbent, closing the gap.

That the lower bound ends up slightly *above* the upper bound is not a bug. The integer cuts have removed
every design already evaluated, so $z^{L}$ is a bound on the designs **not yet tried** — and once that
exceeds the best design in hand, nothing untried can beat it. The search is over. BGW report convergence
in 3 to 5 iterations typically, and in no more than two when $f$ and $g$ are linear; this is a
five-binary problem with 32 possible designs, settled in three NLPs and three MILPs.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4))

ax.plot(history.index, history["z_upper"], marker="o", label=r"upper bound $z^u$ (NLP)")
ax.plot(
    history.index, history["z_lower"], marker="s", label=r"lower bound $z^L$ (MILP)"
)

ax.set_xlabel("outer approximation iteration")
ax.set_ylabel("annualized cost [\\$/m$^2$]")
ax.set_xticks(history.index)
ax.legend(loc="center right", fontsize=11)
plt.show()

The two bounds close on the answer from opposite sides. That picture is the one to keep: it is what a
MINLP solver log is telling you, and a gap that has not closed means the number on the screen is not yet
an answer.

### Checking against a real MINLP solver

Bonmin implements outer approximation and NLP-based branch and bound. Handing it the same model is the
check on everything above — and, in practice, what you would actually do.

In [ ]:
def build_insulation_minlp():
    """The full MINLP, for a solver that can take it whole."""
    m = pyo.ConcreteModel("insulation MINLP")
    m.N = pyo.Set(initialize=list(materials.index))
    m.x = pyo.Var(m.N, domain=pyo.NonNegativeReals, bounds=(0, T_TOTAL))
    m.y = pyo.Var(m.N, domain=pyo.Binary)

    @m.Constraint(m.N)
    def layer_available(m, n):
        return m.x[n] <= materials.loc[n, "tmax"] * m.y[n]

    @m.Constraint()
    def thickness_budget(m):
        return sum(m.x[n] for n in m.N) <= T_TOTAL

    @m.Objective(sense=pyo.minimize)
    def cost(m):
        R = R0 + sum(m.x[n] / materials.loc[n, "k"] for n in m.N)
        return ALPHA / R + BETA * sum(
            materials.loc[n, "a"] * m.y[n] + materials.loc[n, "b"] * m.x[n] for n in m.N
        )

    return m


m_minlp = build_insulation_minlp()
results = pyo.SolverFactory("bonmin").solve(m_minlp)
assert pyo.check_optimal_termination(
    results
), f"bonmin failed: {results.solver.termination_condition}"

print(f"bonmin: cost = {pyo.value(m_minlp.cost):0.4f} $/m^2")
print("        layers = " f"{[n for n in m_minlp.N if pyo.value(m_minlp.y[n]) > 0.5]}")
assert (
    abs(pyo.value(m_minlp.cost) - oa_cost) < 1e-3
), "our outer approximation disagrees with bonmin"

Same answer, to four decimals.

## Where this stops working

Both algorithms in this notebook rest on the same property, and it is worth saying plainly which one.

**Branch and bound** needs the relaxation to be a *valid bound*. Relaxing integrality always is, so §1 is
safe for any MILP.

**Outer approximation** needs $f$ and $g$ to be **convex**, so that a tangent plane underestimates. That
held here, and it was checked rather than assumed. When it does not hold — a concave cost correlation, a
bilinear mixing term — a tangent plane can slice through the feasible region, the "lower bound" is not a
bound, and the method can terminate on the wrong design while reporting a closed gap. DICOPT's
augmented-penalty variant keeps going on nonconvex problems, but it gives up the guarantee to do so.

That is the subject of the next notebook, [](../8/Global-Opt.ipynb), where the relaxation is built to be
valid *without* assuming convexity, and the branching moves from the binaries to the continuous variables.

## Further reading

* Handout: *Integer Programming Algorithms*, and the companion notebook [](../8/MILP.ipynb).
* Biegler, Grossmann & Westerberg (1997), Appendix A: §A.3.2 pp. 757–761 for branch and bound, including
  problem (MIPEX) and Figure A.10; §A.3.4 pp. 763–768 for outer approximation, the integer cut, the
  infeasible-subproblem variant, and the convexity requirement.
* Biegler (2010), §1.3 p. 6, on why NLP algorithms are components of MINLP strategies; §11.1 p. 325 for
  the bilevel structure of §2.
* Duran & Grossmann (1986) for outer approximation; Geoffrion (1972) for generalized Benders
  decomposition; Nemhauser & Wolsey (1988) for branch and bound and cutting planes.
* Postek, Zocca, Gromicho & Kantor, *Hands-On Mathematical Optimization with Python*, Cambridge University
  Press (2025), notebook 6.4, for the insulation application and data.